In [ ]:
import numpy as np
import copy

"""
Sequence Models - DeepLearning.AI Coursera Assignments
  (i) Building an RNN Step by Step
  (ii) Dinosaurus Island - Character-level Language Model
  (iii) Improvise a Jazz Solo with LSTM
  (iv) Emojify with Word Embeddings
  (v) Neural Machine Translation with Attention
  (vi) Trigger Word Detection
"""
# ──--────────────────────────────────────────────────────────────────
# (i) BUILDING A RECURRENT NEURAL NETWORK
# ──--────────────────────────────────────────────────────────────────

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)


# ── Basic RNN ────────────────────────────────────────────────────────────────

def rnn_cell_forward(xt, a_prev, parameters):
    """
    One forward step of a vanilla RNN cell.

    Arguments:
        xt      -- input at time t,  shape (n_x, m)
        a_prev  -- hidden state at t-1, shape (n_a, m)
        parameters -- dict with Wax, Waa, Wya, ba, by

    Returns:
        a_next  -- new hidden state, shape (n_a, m)
        yt_pred -- prediction at t,  shape (n_y, m)
        cache   -- values needed for backprop
    """
    Wax, Waa, Wya = parameters["Wax"], parameters["Waa"], parameters["Wya"]
    ba,  by        = parameters["ba"],  parameters["by"]

    a_next  = np.tanh(Waa @ a_prev + Wax @ xt + ba)   # hidden state
    yt_pred = softmax(Wya @ a_next + by)               # output

    cache = (a_next, a_prev, xt, parameters)
    return a_next, yt_pred, cache


def rnn_forward(x, a0, parameters):
    """
    Forward pass over all T_x time steps.

    Arguments:
        x   -- input data, shape (n_x, m, T_x)
        a0  -- initial hidden state, shape (n_a, m)

    Returns:
        a      -- hidden states for every step, shape (n_a, m, T_x)
        y_pred -- predictions,                  shape (n_y, m, T_x)
        caches -- list of per-step caches
    """
    n_x, m, T_x = x.shape
    n_y, n_a    = parameters["Wya"].shape

    a      = np.zeros((n_a, m, T_x))
    y_pred = np.zeros((n_y, m, T_x))
    caches = []

    a_next = a0
    for t in range(T_x):
        a_next, yt_pred, cache = rnn_cell_forward(x[:, :, t], a_next, parameters)
        a[:, :, t]      = a_next
        y_pred[:, :, t] = yt_pred
        caches.append(cache)

    return a, y_pred, (caches, x)


# ── LSTM ─────────────────────────────────────────────────────────────────────

def lstm_cell_forward(xt, a_prev, c_prev, parameters):
    """
    One forward step of an LSTM cell.

    Gates: forget (f), update (i), output (o)
    Cell candidate: c̃

    Returns: a_next, c_next, yt_pred, cache
    """
    Wf = parameters["Wf"];  bf = parameters["bf"]
    Wi = parameters["Wi"];  bi = parameters["bi"]
    Wc = parameters["Wc"];  bc = parameters["bc"]
    Wo = parameters["Wo"];  bo = parameters["bo"]
    Wy = parameters["Wy"];  by = parameters["by"]

    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    concat = np.vstack((a_prev, xt))        # (n_a + n_x, m)

    ft      = sigmoid(Wf @ concat + bf)     # forget gate
    it      = sigmoid(Wi @ concat + bi)     # update gate
    cct     = np.tanh(Wc @ concat + bc)     # cell candidate
    c_next  = ft * c_prev + it * cct        # cell state
    ot      = sigmoid(Wo @ concat + bo)     # output gate
    a_next  = ot * np.tanh(c_next)          # hidden state

    yt_pred = softmax(Wy @ a_next + by)

    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)
    return a_next, c_next, yt_pred, cache


def lstm_forward(x, a0, parameters):
    """LSTM forward pass over T_x steps."""
    caches = []
    n_x, m, T_x = x.shape
    n_y, n_a     = parameters["Wy"].shape

    a = np.zeros((n_a, m, T_x))
    c = np.zeros((n_a, m, T_x))
    y = np.zeros((n_y, m, T_x))

    a_next = a0
    c_next = np.zeros_like(a0)

    for t in range(T_x):
        a_next, c_next, yt, cache = lstm_cell_forward(x[:, :, t], a_next, c_next, parameters)
        a[:, :, t] = a_next
        c[:, :, t] = c_next
        y[:, :, t] = yt
        caches.append(cache)

    return a, y, c, (caches, x)


# ── Backpropagation ───────────────────────────────────────────────────────────
def rnn_cell_backward(da_next, cache):
    """Backprop through a single vanilla RNN cell."""
    a_next, a_prev, xt, parameters = cache
    Wax, Waa, Wya = parameters["Wax"], parameters["Waa"], parameters["Wya"]

    dtanh  = (1 - a_next ** 2) * da_next   # through tanh

    dxt    = Wax.T @ dtanh
    da_prev= Waa.T @ dtanh
    dWax   = dtanh @ xt.T
    dWaa   = dtanh @ a_prev.T
    dba    = dtanh.sum(axis=1, keepdims=True)

    return {"dxt": dxt, "da_prev": da_prev,
            "dWax": dWax, "dWaa": dWaa, "dba": dba}


def rnn_backward(da, caches):
    """Full RNN backward pass."""
    caches, x = caches
    n_a, m, T_x = da.shape
    n_x, _ = caches[0][2].shape

    dx     = np.zeros((n_x, m, T_x))
    dWax   = np.zeros_like(caches[0][3]["Wax"])
    dWaa   = np.zeros_like(caches[0][3]["Waa"])
    dba    = np.zeros_like(caches[0][3]["ba"])
    da0    = np.zeros((n_a, m))
    da_prevt = np.zeros_like(da0)

    for t in reversed(range(T_x)):
        grads = rnn_cell_backward(da[:, :, t] + da_prevt, caches[t])
        dx[:, :, t] = grads["dxt"]
        dWax  += grads["dWax"]
        dWaa  += grads["dWaa"]
        dba   += grads["dba"]
        da_prevt = grads["da_prev"]

    da0 = da_prevt
    return {"dx": dx, "da0": da0, "dWax": dWax, "dWaa": dWaa, "dba": dba}


def lstm_cell_backward(da_next, dc_next, cache):
    """Backprop through a single LSTM cell."""
    (a_next, c_next, a_prev, c_prev,
     ft, it, cct, ot, xt, parameters) = cache

    n_x, m = xt.shape
    n_a, _ = a_next.shape

    # Gradients through gates
    dot  = da_next * np.tanh(c_next)  * ot * (1 - ot)
    dcct = (dc_next * it + ot * (1 - np.tanh(c_next)**2) * it * da_next) * (1 - cct**2)
    dit  = (dc_next * cct + ot * (1 - np.tanh(c_next)**2) * cct * da_next) * it * (1 - it)
    dft  = (dc_next * c_prev + ot * (1 - np.tanh(c_next)**2) * c_prev * da_next) * ft * (1 - ft)

    concat = np.vstack((a_prev, xt))

    dWf  = dft  @ concat.T;  dbf = dft.sum(axis=1, keepdims=True)
    dWi  = dit  @ concat.T;  dbi = dit.sum(axis=1, keepdims=True)
    dWc  = dcct @ concat.T;  dbc = dcct.sum(axis=1, keepdims=True)
    dWo  = dot  @ concat.T;  dbo = dot.sum(axis=1, keepdims=True)

    Wf, Wi, Wc, Wo = (parameters[k] for k in ("Wf", "Wi", "Wc", "Wo"))
    da_prev = (Wf[:, :n_a].T @ dft  + Wi[:, :n_a].T @ dit  +
               Wc[:, :n_a].T @ dcct + Wo[:, :n_a].T @ dot)
    dxt     = (Wf[:, n_a:].T @ dft  + Wi[:, n_a:].T @ dit  +
               Wc[:, n_a:].T @ dcct + Wo[:, n_a:].T @ dot)
    dc_prev = dc_next * ft + ot * (1 - np.tanh(c_next)**2) * ft * da_next

    return {"dxt": dxt, "da_prev": da_prev, "dc_prev": dc_prev,
            "dWf": dWf, "dbf": dbf, "dWi": dWi, "dbi": dbi,
            "dWc": dWc, "dbc": dbc, "dWo": dWo, "dbo": dbo}


def lstm_backward(da, caches):
    """Full LSTM backward pass."""
    caches_list, x = caches
    n_a, m, T_x = da.shape
    n_x = x.shape[0]

    dx = np.zeros((n_x, m, T_x))
    da0 = np.zeros((n_a, m))
    dc0 = np.zeros((n_a, m))

    grads_sum = {k: np.zeros_like(caches_list[0][9][k])
                 for k in ("Wf", "Wi", "Wc", "Wo")}
    bias_grads = {k: np.zeros_like(caches_list[0][9][k])
                  for k in ("bf", "bi", "bc", "bo")}

    da_prevt = np.zeros_like(da0)
    dc_prevt = np.zeros_like(dc0)

    for t in reversed(range(T_x)):
        grads = lstm_cell_backward(da[:, :, t] + da_prevt, dc_prevt, caches_list[t])
        dx[:, :, t] = grads["dxt"]
        for k in ("Wf", "Wi", "Wc", "Wo"):
            grads_sum[k] += grads["d" + k]
        for k in ("bf", "bi", "bc", "bo"):
            bias_grads[k] += grads["d" + k]
        da_prevt = grads["da_prev"]
        dc_prevt = grads["dc_prev"]

    da0 = da_prevt
    return {"dx": dx, "da0": da0, **grads_sum, **bias_grads}


# ──--────────────────────────────────────────────────────────────────
# (ii) DINOSAURUS ISLAND — CHARACTER-LEVEL LANGUAGE MODEL
# ──--────────────────────────────────────────────────────────────────
def clip(gradients, maxValue):
    """Gradient clipping to avoid exploding gradients."""
    grads = copy.deepcopy(gradients)
    for key in ["dWaa", "dWax", "dWya", "db", "dby"]:
        grads[key] = np.clip(grads[key], -maxValue, maxValue)
    return grads


def sample(parameters, char_to_ix, seed=0):
    """
    Sample a sequence of characters from the RNN language model.

    Returns:
        indices -- list of character indices sampled
    """
    Waa, Wax, Wya, by, b = (parameters[k]
        for k in ("Waa", "Wax", "Wya", "by", "b"))
    vocab_size = by.shape[0]
    n_a = Waa.shape[1]

    x = np.zeros((vocab_size, 1))
    a_prev = np.zeros((n_a, 1))

    indices = []
    idx = -1
    counter = 0
    newline_character = char_to_ix.get("\n", 0)

    while idx != newline_character and counter < 50:
        a   = np.tanh(Wax @ x + Waa @ a_prev + b)
        z   = Wya @ a + by
        y   = softmax(z)

        np.random.seed(counter + seed)
        idx = np.random.choice(range(vocab_size), p=y.ravel())
        indices.append(idx)

        x = np.zeros((vocab_size, 1))
        x[idx] = 1
        a_prev = a
        counter += 1

    return indices


def optimize(X, Y, a_prev, parameters, learning_rate=0.01):
    """
    One training step: forward → loss → backward → clip → update.

    Returns: loss, updated parameters, last hidden state
    """
    # Forward pass
    loss = 0
    a = {}
    a[-1] = a_prev.copy()
    caches = []
    x = {}
    n_a = parameters["Waa"].shape[0]
    vocab_size = parameters["Wya"].shape[0]

    y_hat = {}
    for t, (ix_x, ix_y) in enumerate(zip(X, Y)):
        x[t] = np.zeros((vocab_size, 1))
        if ix_x is not None:
            x[t][ix_x] = 1

        a[t], _, cache = rnn_cell_forward(x[t], a[t - 1], parameters)
        y_hat[t] = softmax(parameters["Wya"] @ a[t] + parameters["by"])
        loss -= np.log(y_hat[t][ix_y, 0])
        caches.append(cache)

    # Backward pass (manual, character-level)
    dWax = np.zeros_like(parameters["Wax"])
    dWaa = np.zeros_like(parameters["Waa"])
    dWya = np.zeros_like(parameters["Wya"])
    db   = np.zeros_like(parameters["b"])
    dby  = np.zeros_like(parameters["by"])
    da_next = np.zeros_like(a[0])

    for t in reversed(range(len(X))):
        dy = y_hat[t].copy()
        dy[Y[t]] -= 1
        dWya += dy @ a[t].T
        dby  += dy
        da   = parameters["Wya"].T @ dy + da_next
        daraw = (1 - a[t] ** 2) * da
        db   += daraw
        dWax += daraw @ x[t].T
        dWaa += daraw @ a[t - 1].T
        da_next = parameters["Waa"].T @ daraw

    gradients = {"dWax": dWax, "dWaa": dWaa, "dWya": dWya, "db": db, "dby": dby}
    gradients = clip(gradients, maxValue=5)

    # Parameter update (SGD)
    for key in ("Wax", "Waa", "Wya", "b", "by"):
        parameters[key] -= learning_rate * gradients["d" + key]

    return loss, parameters, a[len(X) - 1]


def model(data_text, ix_to_char, char_to_ix, num_iterations=35000,
          n_a=50, dino_names=7, vocab_size=27, verbose=False):
    """
    Train the Dino-name RNN and print sampled names every 2000 steps.
    """
    n_x, n_y = vocab_size, vocab_size
    parameters = {
        "Wax": np.random.randn(n_a, n_x) * 0.01,
        "Waa": np.random.randn(n_a, n_a) * 0.01,
        "Wya": np.random.randn(n_y, n_a) * 0.01,
        "b":   np.zeros((n_a, 1)),
        "by":  np.zeros((n_y, 1)),
    }

    examples = [x.lower().strip() for x in data_text.split("\n") if x.strip()]
    np.random.seed(0)
    np.random.shuffle(examples)
    a_prev = np.zeros((n_a, 1))
    best_loss = float("inf")

    for j in range(num_iterations):
        idx   = j % len(examples)
        chars = [None] + [char_to_ix[c] for c in examples[idx]]
        X, Y  = chars[:-1], chars[1:]

        curr_loss, parameters, a_prev = optimize(X, Y, a_prev, parameters)

        if verbose and j % 2000 == 0:
            print(f"Iteration {j:5d}, Loss: {curr_loss:.4f}")
            for s in range(dino_names):
                idxs = sample(parameters, char_to_ix, seed=s)
                name = "".join(ix_to_char[i] for i in idxs if ix_to_char[i] != "\n")
                print(f"  {name.capitalize()}")
            print()

    return parameters


# ──--────────────────────────────────────────────────────────────────
# (iii) JAZZ IMPROVISATION WITH LSTM  withe Keras API
# ──--────────────────────────────────────────────────────────────────
# NOTE: Full Keras model shown; requires tensorflow>=2.x

def build_jazz_model(Tx, n_a, n_values):
    """
    LSTM model for jazz solo generation.

    Architecture: Input → LSTM(n_a) → Dense(n_values, softmax)
    Uses shared layers so weights are reused across time steps during inference.
    """
    try:
        from tensorflow.keras.layers import (Dense, Input, LSTM,
                                              Activation, Lambda)
        from tensorflow.keras.models import Model
        import tensorflow.keras.backend as K

        reshapor  = Lambda(lambda x: K.reshape(x, (K.shape(x)[0], 1, n_values)))
        LSTM_cell = LSTM(n_a, return_state=True)
        densor    = Dense(n_values, activation="softmax")

        # Training model
        X    = Input(shape=(Tx, n_values))
        a0   = Input(shape=(n_a,), name="a0")
        c0   = Input(shape=(n_a,), name="c0")
        a, c = a0, c0
        outputs = []

        for t in range(Tx):
            x          = Lambda(lambda z: z[:, t, :])(X)
            x          = reshapor(x)
            a, _, c    = LSTM_cell(x, initial_state=[a, c])
            out        = densor(a)
            outputs.append(out)

        model = Model(inputs=[X, a0, c0], outputs=outputs)
        return model, LSTM_cell, densor, reshapor

    except ImportError:
        print("TensorFlow not installed — returning None")
        return None, None, None, None


def music_inference_model(LSTM_cell, densor, reshapor, n_a, n_values, Ty=100):
    """Build inference model that generates one note at a time."""
    try:
        from tensorflow.keras.layers import Input, Lambda
        from tensorflow.keras.models import Model
        import tensorflow.keras.backend as K

        x0      = Input(shape=(1, n_values))
        a0      = Input(shape=(n_a,), name="a0")
        c0      = Input(shape=(n_a,), name="c0")
        a, c, x = a0, c0, x0
        outputs = []

        for _ in range(Ty):
            a, _, c = LSTM_cell(x, initial_state=[a, c])
            out     = densor(a)
            outputs.append(out)
            x = reshapor(out)

        return Model(inputs=[x0, a0, c0], outputs=outputs)
    except ImportError:
        return None


#──--────────────────────────────────────────────────────────────────
# (iv) EMOJIFY — WORD EMBEDDINGS + LSTM
# ──--────────────────────────────────────────────────────────────────

def read_glove_vecs(glove_file):
    """Load pre-trained GloVe embeddings into a dict and index lookup."""
    words_to_idx, idx_to_words, word_to_vec = {}, {}, {}
    with open(glove_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            word = parts[0]
            words_to_idx[word] = i
            idx_to_words[i]    = word
            word_to_vec[word]  = np.array(parts[1:], dtype=np.float64)
    return words_to_idx, idx_to_words, word_to_vec


def sentence_to_avg(sentence, word_to_vec_map):
    """
    Average GloVe embeddings of all words in a sentence.
    Used in the simpler (non-LSTM) Emojify-v1 baseline.
    """
    words  = sentence.lower().split()
    any_w  = next(iter(word_to_vec_map))
    avg    = np.zeros(word_to_vec_map[any_w].shape)
    total  = 0
    for w in words:
        if w in word_to_vec_map:
            avg  += word_to_vec_map[w]
            total += 1
    if total > 0:
        avg /= total
    return avg


def softmax_forward(z):
    return softmax(z)


def emojify_forward(X, Y, word_to_vec_map, W, b):
    """
    Simple softmax emoji classifier forward pass.
    X: list of sentences, Y: list of labels (0-4)
    """
    m = len(X)
    n_y, _ = W.shape
    cost = 0
    for i in range(m):
        avg = sentence_to_avg(X[i], word_to_vec_map)
        z   = W @ avg + b
        a   = softmax(z)
        cost -= np.log(a[Y[i]])
    return cost / m


def sentences_to_indices(X, word_to_index, max_len):
    """Convert list of sentences to padded index matrix (m, max_len)."""
    m = len(X)
    X_idx = np.zeros((m, max_len), dtype=int)
    for i, sentence in enumerate(X):
        words = sentence.lower().split()
        for j, w in enumerate(words[:max_len]):
            X_idx[i, j] = word_to_index.get(w, 0)
    return X_idx


def pretrained_embedding_layer(word_to_vec_map, word_to_index):
    """
    Build a Keras Embedding layer initialised with GloVe weights.
    Returns the layer (not yet compiled into a model).
    """
    try:
        from tensorflow.keras.layers import Embedding
        vocab_len  = len(word_to_index) + 1
        emb_dim    = next(iter(word_to_vec_map.values())).shape[0]
        emb_matrix = np.zeros((vocab_len, emb_dim))
        for w, idx in word_to_index.items():
            if w in word_to_vec_map:
                emb_matrix[idx] = word_to_vec_map[w]
        layer = Embedding(vocab_len, emb_dim, trainable=False)
        layer.build((None,))
        layer.set_weights([emb_matrix])
        return layer
    except ImportError:
        return None


def build_emojify_model(input_shape, word_to_vec_map, word_to_index):
    """
    LSTM-based emoji classifier (Emojify-v2).
    input_shape: (max_len,)
    """
    try:
        from tensorflow.keras.layers import (Input, LSTM, Dense,
                                              Dropout, Bidirectional)
        from tensorflow.keras.models import Model

        sentence_indices = Input(shape=input_shape, dtype="int32")
        embedding_layer  = pretrained_embedding_layer(word_to_vec_map, word_to_index)
        embeddings        = embedding_layer(sentence_indices)

        X = LSTM(128, return_sequences=True)(embeddings)
        X = Dropout(0.5)(X)
        X = LSTM(128)(X)
        X = Dropout(0.5)(X)
        X = Dense(5, activation="softmax")(X)

        return Model(inputs=sentence_indices, outputs=X)
    except ImportError:
        return None


# ──--────────────────────────────────────────────────────────────────
# (v) NEURAL MACHINE TRANSLATION WITH ATTENTION
#──--────────────────────────────────────────────────────────────────
def softmax_attention(x):
    """Softmax over axis=-1 for attention weights."""
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def one_step_attention(a, s_prev, repeator, concatenator,
                       densor1, densor2, activator, dotor):
    """
    Attention mechanism — one decoder step.

    a:      encoder hidden states,  (m, Tx, 2*n_a)
    s_prev: previous decoder state, (m, n_s)

    Returns context vector c of shape (m, 1, 2*n_a)
    """
    # Repeat decoder state Tx times
    s_prev_rep = repeator(s_prev)           # (m, Tx, n_s)
    concat     = concatenator([a, s_prev_rep])
    e          = densor1(concat)
    e          = activator(e)
    e          = densor2(e)                 # (m, Tx, 1) — energies
    alphas     = activator(e)               # attention weights (softmax applied inside layer)
    context    = dotor([alphas, a])         # (m, 1, 2*n_a)
    return context


def build_nmt_attention_model(Tx, Ty, n_a, n_s, human_vocab_size, machine_vocab_size):
    """
    Seq2Seq + Bahdanau attention for date translation.

    Encoder: Bidirectional LSTM
    Decoder: LSTM with attention over encoder outputs
    """
    try:
        from tensorflow.keras.layers import (
            Bidirectional, Concatenate, Dense, Dot,
            Input, LSTM, RepeatVector, Activation
        )
        from tensorflow.keras.models import Model

        # Shared layers
        repeator      = RepeatVector(Tx)
        concatenator  = Concatenate(axis=-1)
        densor1       = Dense(10, activation="tanh")
        densor2       = Dense(1, activation="relu")
        activator     = Activation(softmax_attention, name="attention_weights")
        dotor         = Dot(axes=1)
        post_lstm     = LSTM(n_s, return_state=True)
        output_layer  = Dense(machine_vocab_size, activation="softmax")

        # Inputs
        X    = Input(shape=(Tx, human_vocab_size))
        s0   = Input(shape=(n_s,), name="s0")
        c0   = Input(shape=(n_s,), name="c0")
        s, c = s0, c0

        # Encoder
        a = Bidirectional(LSTM(n_a, return_sequences=True))(X)

        # Decoder
        outputs = []
        for _ in range(Ty):
            context       = one_step_attention(a, s, repeator, concatenator,
                                               densor1, densor2, activator, dotor)
            s, _, c       = post_lstm(context, initial_state=[s, c])
            out           = output_layer(s)
            outputs.append(out)

        return Model(inputs=[X, s0, c0], outputs=outputs)
    except ImportError:
        return None


# ──--────────────────────────────────────────────────────────────────
# (vi) TRIGGER WORD DETECTION
# ──--────────────────────────────────────────────────────────────────

def is_overlapping(segment_time, previous_segments):
    """Return True if [l,r] overlaps any already-placed segment."""
    l, r = segment_time
    return any(not (r < ps[0] or l > ps[1]) for ps in previous_segments)


def insert_audio_clip(background, audio_clip, previous_segments):
    """
    Randomly place audio_clip into background without overlapping.
    Returns (new_background, segment_time) or raises RuntimeError.
    """
    seg_ms = len(audio_clip)
    for _ in range(5):
        seg_start = np.random.randint(0, len(background) - seg_ms)
        seg_end   = seg_start + seg_ms
        candidate = (seg_start, seg_end)
        if not is_overlapping(candidate, previous_segments):
            new_bg = background.overlay(audio_clip, position=seg_start)
            return new_bg, candidate
    raise RuntimeError("Could not place clip without overlap after 5 attempts")


def insert_ones(y, segment_end_ms, Ty=1375, trigger_len=50):
    """
    After a trigger word ends at segment_end_ms, set 50 consecutive
    label frames to 1 in the label tensor y.
    """
    segment_end_y = int(segment_end_ms * Ty / 10000)
    for i in range(segment_end_y + 1, segment_end_y + trigger_len + 1):
        if i < Ty:
            y[0, i, 0] = 1
    return y


def create_training_example(background, activates, negatives, Ty=1375):
    """
    Compose one training audio clip:
      - random trigger words → label 1 for 50 frames after each
      - random non-trigger fillers
    Returns (x_spectrogram, y_labels).
    """
    y = np.zeros((1, Ty, 1))
    previous_segments = []

    # Insert random activates
    n_activates = np.random.randint(0, 5)
    for _ in range(n_activates):
        clip = activates[np.random.randint(len(activates))]
        try:
            background, seg = insert_audio_clip(background, clip, previous_segments)
            previous_segments.append(seg)
            y = insert_ones(y, seg[1])
        except RuntimeError:
            pass

    # Insert random negatives
    n_negatives = np.random.randint(0, 3)
    for _ in range(n_negatives):
        clip = negatives[np.random.randint(len(negatives))]
        try:
            background, seg = insert_audio_clip(background, clip, previous_segments)
            previous_segments.append(seg)
        except RuntimeError:
            pass

    return background, y


def build_trigger_word_model(input_shape):
    """
    1-D Conv → 2 × GRU → Dense(1, sigmoid) for trigger-word detection.
    input_shape: (T_x, n_freq)
    """
    try:
        from tensorflow.keras.layers import (
            Conv1D, Dense, Dropout, GRU,
            Input, BatchNormalization
        )
        from tensorflow.keras.models import Model

        X_input = Input(shape=input_shape)

        X = Conv1D(196, kernel_size=15, strides=4)(X_input)
        X = BatchNormalization()(X)
        X = Activation("relu")(X)
        X = Dropout(0.8)(X)

        X = GRU(128, return_sequences=True)(X)
        X = Dropout(0.8)(X)
        X = BatchNormalization()(X)

        X = GRU(128, return_sequences=True)(X)
        X = Dropout(0.8)(X)
        X = BatchNormalization()(X)
        X = Dropout(0.8)(X)

        X = Dense(1, activation="sigmoid")(X)
        return Model(inputs=X_input, outputs=X)
    except (ImportError, NameError):
        return None


def detect_triggerword(filename, model, threshold=0.5):
    """
    Run trigger-word detection on a .wav file.
    Prints timestamp of each detection.
    Requires: librosa, scipy
    """
    try:
        import librosa

        y, sr  = librosa.load(filename, sr=44100)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        x      = chroma.T[np.newaxis, ...]       # (1, T, n_freq)
        pred   = model.predict(x)[0, :, 0]

        trigger_times = []
        for i, p in enumerate(pred):
            if p > threshold:
                t = i / len(pred) * (len(y) / sr)
                trigger_times.append(t)
        return trigger_times
    except ImportError:
        print("librosa not available")
        return []


# ──--────────────────────────────────────────────────────────────────
# QUICK TEST
# ──--────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    np.random.seed(42)
    print("=" * 60)
    print("Smoke-testing RNN / LSTM forward passes")
    print("=" * 60)

    # Dimensions
    n_x, n_a, n_y, m, T_x = 3, 5, 2, 4, 6

    # ── RNN cell ──────────────────────────────────────────────────────────────
    rnn_params = {
        "Wax": np.random.randn(n_a, n_x),
        "Waa": np.random.randn(n_a, n_a),
        "Wya": np.random.randn(n_y, n_a),
        "ba":  np.zeros((n_a, 1)),
        "by":  np.zeros((n_y, 1)),
    }
    xt     = np.random.randn(n_x, m)
    a_prev = np.random.randn(n_a, m)
    a_next, yt, cache = rnn_cell_forward(xt, a_prev, rnn_params)
    print(f"RNN cell  → a_next: {a_next.shape}, yt: {yt.shape}")

    x = np.random.randn(n_x, m, T_x)
    a0 = np.zeros((n_a, m))
    a_all, y_all, rnn_caches = rnn_forward(x, a0, rnn_params)
    print(f"RNN fwd   → a: {a_all.shape}, y: {y_all.shape}")

    # ── LSTM cell ─────────────────────────────────────────────────────────────
    lstm_params = {
        "Wf": np.random.randn(n_a, n_a + n_x), "bf": np.zeros((n_a, 1)),
        "Wi": np.random.randn(n_a, n_a + n_x), "bi": np.zeros((n_a, 1)),
        "Wc": np.random.randn(n_a, n_a + n_x), "bc": np.zeros((n_a, 1)),
        "Wo": np.random.randn(n_a, n_a + n_x), "bo": np.zeros((n_a, 1)),
        "Wy": np.random.randn(n_y, n_a),        "by": np.zeros((n_y, 1)),
    }
    c_prev = np.random.randn(n_a, m)
    an, cn, yp, lcache = lstm_cell_forward(xt, a_prev, c_prev, lstm_params)
    print(f"LSTM cell → a_next: {an.shape}, c_next: {cn.shape}")

    a_l, y_l, c_l, lstm_caches = lstm_forward(x, a0, lstm_params)
    print(f"LSTM fwd  → a: {a_l.shape}, y: {y_l.shape}, c: {c_l.shape}")

    # ── Backward ──────────────────────────────────────────────────────────────
    da = np.random.randn(*a_all.shape)
    rnn_grads = rnn_backward(da, rnn_caches)
    print(f"RNN bwd   → dx: {rnn_grads['dx'].shape}")

    lstm_grads = lstm_backward(da, lstm_caches)
    print(f"LSTM bwd  → dx: {lstm_grads['dx'].shape}")

    print("\nAll forward / backward checks passed ✓")

Smoke-testing RNN / LSTM forward passes
RNN cell  → a_next: (5, 4), yt: (2, 4)
RNN fwd   → a: (5, 4, 6), y: (2, 4, 6)
LSTM cell → a_next: (5, 4), c_next: (5, 4)
LSTM fwd  → a: (5, 4, 6), y: (2, 4, 6), c: (5, 4, 6)
RNN bwd   → dx: (3, 4, 6)
LSTM bwd  → dx: (3, 4, 6)

All forward / backward checks passed ✓
